In [ ]:
import os
import json
import re
import numpy as np
import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoConfig, AutoModel
from peft import PeftModel

In [ ]:
MODEL_DIR = "/content/light_best_model"   # hoặc đường dẫn tới thư mục đã unzip
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
with open(os.path.join(MODEL_DIR, "export_meta.json"), "r", encoding="utf-8") as f:
    meta = json.load(f)

MODEL_NAME = meta["model_name"]
MAX_LENGTH = meta["max_length"]
HEAD_DROPOUT = meta["head_dropout"]
TARGET_COLS = meta["target_cols"]
GRA_FEATURE_COLS = meta["gra_feature_cols"]

gra_feat_mean = np.array(meta["gra_feat_mean"], dtype=np.float32)
gra_feat_std = np.array(meta["gra_feat_std"], dtype=np.float32)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


In [ ]:
def extract_grammar_features(text: str) -> np.ndarray:
    text = str(text).strip()

    words = re.findall(r"\b[\w']+\b", text)
    word_count = max(len(words), 1)

    sentence_candidates = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentence_candidates if s.strip()]
    sentence_count = max(len(sentences), 1)

    avg_sentence_len = word_count / sentence_count

    sent_word_counts = []
    for s in sentences:
        sw = re.findall(r"\b[\w']+\b", s)
        sent_word_counts.append(len(sw))

    if len(sent_word_counts) == 0:
        sent_word_counts = [word_count]

    short_sentence_ratio = np.mean([c < 8 for c in sent_word_counts])
    long_sentence_ratio = np.mean([c > 25 for c in sent_word_counts])

    punct_count = len(re.findall(r"[,:;()\-\"]", text))
    punct_density = punct_count / word_count

    repeated_punct_count = len(re.findall(r"([!?.,;:])\1+", text))
    repeated_punct_ratio = repeated_punct_count / sentence_count

    lowercase_sent_starts = 0
    for s in sentences:
        s = s.strip()
        if len(s) > 0 and s[0].islower():
            lowercase_sent_starts += 1
    lowercase_sent_start_ratio = lowercase_sent_starts / sentence_count

    lowercase_i_count = len(re.findall(r"\bi\b", text))
    lowercase_i_ratio = lowercase_i_count / word_count

    repeated_word_count = 0
    lower_words = [w.lower() for w in words]
    for i in range(1, len(lower_words)):
        if lower_words[i] == lower_words[i - 1]:
            repeated_word_count += 1
    repeated_word_ratio = repeated_word_count / word_count

    missing_terminal_punct = 0.0 if re.search(r"[.!?]\s*$", text) else 1.0

    feats = np.array([
        float(word_count),
        float(sentence_count),
        float(avg_sentence_len),
        float(short_sentence_ratio),
        float(long_sentence_ratio),
        float(punct_density),
        float(repeated_punct_ratio),
        float(lowercase_sent_start_ratio),
        float(lowercase_i_ratio),
        float(repeated_word_ratio),
        float(missing_terminal_punct),
    ], dtype=np.float32)

    feats = (feats - gra_feat_mean) / gra_feat_std
    return feats

In [ ]:
class QwenForIELTSMultiTask(nn.Module):
    def __init__(self, model_name: str, tokenizer, model_dir: str, head_dropout: float = 0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.pad_token_id = tokenizer.pad_token_id

        load_dtype = (
            torch.bfloat16
            if torch.cuda.is_available() and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported()
            else (torch.float16 if torch.cuda.is_available() else torch.float32)
        )

        base_model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=load_dtype,
        )

        self.backbone = PeftModel.from_pretrained(base_model, model_dir)
        self.backbone.config.pad_token_id = tokenizer.pad_token_id
        self.backbone.config.use_cache = False

        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(head_dropout)

        self.head_tr = nn.Linear(hidden_size, 1)
        self.head_cc = nn.Linear(hidden_size, 1)
        self.head_lr = nn.Linear(hidden_size, 1)

        self.gra_feat_dim = len(GRA_FEATURE_COLS)
        self.gra_mlp = nn.Sequential(
            nn.Linear(hidden_size + self.gra_feat_dim, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )

        self.reg_activation = nn.Sigmoid()

    def _last_token_pool(self, hidden_states, attention_mask):
        last_token_idx = attention_mask.sum(dim=1) - 1
        last_token_idx = last_token_idx.clamp(min=0)
        batch_idx = torch.arange(hidden_states.size(0), device=hidden_states.device)
        pooled = hidden_states[batch_idx, last_token_idx]
        return pooled

    def forward(self, input_ids=None, attention_mask=None, gra_features=None):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state
        pooled = self._last_token_pool(hidden_states, attention_mask)
        pooled = self.dropout(pooled)

        tr = self.reg_activation(self.head_tr(pooled))
        cc = self.reg_activation(self.head_cc(pooled))
        lr = self.reg_activation(self.head_lr(pooled))

        gra_features = gra_features.to(device=pooled.device, dtype=pooled.dtype)
        gra_input = torch.cat([pooled, gra_features], dim=1)
        gra = self.reg_activation(self.gra_mlp(gra_input))

        logits = torch.cat([tr, cc, lr, gra], dim=1)
        return logits


model = QwenForIELTSMultiTask(
    model_name=MODEL_NAME,
    tokenizer=tokenizer,
    model_dir=MODEL_DIR,
    head_dropout=HEAD_DROPOUT,
)

heads = torch.load(os.path.join(MODEL_DIR, "custom_heads.pt"), map_location="cpu")
model.head_tr.load_state_dict(heads["head_tr"])
model.head_cc.load_state_dict(heads["head_cc"])
model.head_lr.load_state_dict(heads["head_lr"])
model.gra_mlp.load_state_dict(heads["gra_mlp"])

model.to(DEVICE)
model.eval()

print("Loaded model from:", MODEL_DIR)
print("Using device:", DEVICE)

In [ ]:
def round_to_half(x):
    return np.round(x * 2) / 2


def predict_ielts_scores(prompt: str, essay: str, round_band: bool = True):
    text = f"[PROMPT]\n{str(prompt).strip()}\n\n[ESSAY]\n{str(essay).strip()}"

    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors="pt",
    )

    gra_features = extract_grammar_features(essay)
    gra_features = torch.tensor(gra_features, dtype=torch.float32).unsqueeze(0)

    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    gra_features = gra_features.to(DEVICE)

    with torch.no_grad():
        logits = model(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            gra_features=gra_features,
        )

    scores = logits.squeeze(0).detach().float().cpu().numpy() * 9.0
    scores = np.clip(scores, 0.0, 9.0)

    raw_scores = {
        "TR": float(scores[0]),
        "CC": float(scores[1]),
        "LR": float(scores[2]),
        "GRA": float(scores[3]),
        "Overall": float(np.mean(scores)),
    }

    if round_band:
        rounded = {k: float(round_to_half(v)) for k, v in raw_scores.items()}
        return {
            "raw_scores": raw_scores,
            "rounded_scores": rounded,
        }

    return raw_scores

In [ ]:
prompt = "Some people think that children should begin formal education at a very early age. To what extent do you agree or disagree?"

essay = """
In my opinion, children should not begin formal education too early. Although learning basic skills at a young age can be helpful,
forcing children into an academic environment too soon may create stress and reduce their interest in learning.

First, young children learn effectively through play and social interaction. Activities such as drawing, storytelling and group games
help them develop creativity, communication and emotional intelligence. If formal schooling starts too early, these natural forms of
learning may be replaced by pressure to perform academically.

Second, early formal education can negatively affect mental well-being. Some children may struggle to adapt to strict schedules and
classroom discipline, which can lead to anxiety or low confidence. A more balanced approach would allow children to mature emotionally
before facing serious academic demands.

In conclusion, while early exposure to learning can be beneficial, I believe formal education should not start too early because
children need time to develop socially and emotionally first.
"""

result = predict_ielts_scores(prompt, essay, round_band=True)
print(result)